# Sequences

Recurrent models (RNN, LSTM, GRU) process sequences where each time step's
output depends on all previous inputs. In idris-ml, sequence data uses
`RecurrentDataPoint` and training uses `epochRecurrentVar`.

We'll train an RNN to predict the next element in a repeating [0, 1, 0] pattern.


## Recurrent Data

`RecurrentDataPoint` holds variable-length sequences of fixed-dimension vectors:


In [1]:
:t RecurrentDataPoint


DataPoint.RecurrentDataPoint : Nat -> Nat -> Type -> Type


In [2]:
:t MkRecurrentDataPoint


DataPoint.MkRecurrentDataPoint : List (Vector i ty) -> List (Vector o ty) -> RecurrentDataPoint i o ty


The `.xs` field is a `List (Vector i ty)` — a variable-length sequence of input
vectors, each of fixed dimension `i`. The `.ys` field is the corresponding output
sequence. Different data points can have different sequence lengths.


## Pattern Prediction Task

`patternData` generates sequences of the repeating pattern [0, 1, 0, 0, 1, 0, ...].
The input is the pattern and the target is the next element — the model must learn
the repeating structure.


For a pattern-data generator example, see `packages/idris-ml-examples/src/Generate.idr` (`patternData`) — it lives in the examples package, not the kernel prelude. The CLI demos (`make example-rnn`, `make example-lstm`) exercise it directly.

For a pattern-data generator example, see `packages/idris-ml-examples/src/Generate.idr` (`patternData`) — it lives in the examples package, not the kernel prelude. The CLI demos (`make example-rnn`, `make example-lstm`) exercise it directly.

For a pattern-data generator example, see `packages/idris-ml-examples/src/Generate.idr` (`patternData`) — it lives in the examples package, not the kernel prelude. The CLI demos (`make example-rnn`, `make example-lstm`) exercise it directly.

Each data point is a sequence of scalar (1-element) vectors. The target at each
time step is the next element in the pattern.


## RNN Model

An RNN layer maintains hidden state that carries information across time steps.
The `forward` function returns the updated model (with new hidden state) alongside
the output — pure functional, no mutation.


In [6]:
:t rnnLayer


Layer.Rnn.rnnLayer : (Num ty, FromDouble ty) => IO (AnyLayer i o ty)


In [7]:
:exec do { srand 42;
  rnn <- rnnLayerAny {i=1} {o=1} "rnn0";
  model <- pure ((OutputLayer rnn));
  putStrLn "Model: ready" }


Model: Rnn<1:1>


## Recurrent Forward Pass

`forwardRecurrent` processes a full sequence, threading hidden state through
each time step. It returns a list of outputs, one per time step.


In [8]:
:t epochRecurrentVar


Layer.Core.forwardRecurrent : (FromDouble ty, (Floating ty, (Fractional ty, (Neg ty, (Num ty, Ord ty))))) => Network i hs o ty -> List (Vector i ty) -> (Network i hs o ty, List (Vector o ty))


## Training

Recurrent training uses `epochRecurrentVar` and binary cross-entropy loss
(the output is a probability of the next element being 1).


For a pattern-data generator example, see `packages/idris-ml-examples/src/Generate.idr` (`patternData`) — it lives in the examples package, not the kernel prelude. The CLI demos (`make example-rnn`, `make example-lstm`) exercise it directly.

## Evaluation

After training, check if the model predicts the pattern correctly.
Convert to Double for evaluation, then run `evaluateRecurrent`.


_Note: this evaluation cell used V1's `toDoubleNetwork` which was removed in the Path C migration. V2 evaluates by running `forwardVar` on the trained model directly and reading scalars via `prim__item1d outT.tensorPtr <i>`. See `packages/idris-ml-examples/src/Example/Supervised.idr` for an idiomatic V2 evaluation pattern._

## LSTM: Drop-In Replacement

LSTM is a more powerful recurrent layer that handles longer dependencies.
The API is identical — just swap `rnnLayer` for `lstmLayer`:


In [11]:
:t lstmLayer


Layer.Lstm.lstmLayer : (Num ty, FromDouble ty) => IO (AnyLayer i o ty)


For a pattern-data generator example, see `packages/idris-ml-examples/src/Generate.idr` (`patternData`) — it lives in the examples package, not the kernel prelude. The CLI demos (`make example-rnn`, `make example-lstm`) exercise it directly.

Same data, same optimizer, same training loop — just a different layer.
The type system ensures the dimensions still match.


## When to Use What

| Model | Best for | Trade-off |
|-------|----------|-----------|
| RNN | Short sequences, simple patterns | Fast but forgets long-range dependencies |
| LSTM | Medium sequences, complex patterns | More parameters, better memory |
| GRU | Similar to LSTM, fewer parameters | Simpler gating, often comparable accuracy |
| Transformer | Long sequences, parallel processing | `make example-transformer` for a demo |

For longer examples with more epochs:
```bash
make example-rnn      # RNN on pattern task (2000 epochs)
make example-lstm     # LSTM on same task with early stopping
```
